In [ ]:
# import pandas as pd
# import os

# # === 入力ファイルパス ===
# fixation_csv_path = "./exported_csv/fixation_counts/IVT/fixation_with_AOI_labels/fixation_AOI_label_001-001-0.csv"
# timepoints_csv_path = "./exported_csv/sampling_df_add_kukan/with_timepoints_sampling_df_id001-001.csv"

# # === 出力先フォルダ ===
# output_dir = "exported_csv/fixation_sliced_by_time"
# os.makedirs(output_dir, exist_ok=True)

# # === ファイル存在確認 ===
# if not os.path.exists(fixation_csv_path):
#     print(f"⚠️ ファイルが存在しません: {fixation_csv_path}")
#     exit(1)

# if not os.path.exists(timepoints_csv_path):
#     print(f"⚠️ ファイルが存在しません: {timepoints_csv_path}")
#     exit(1)

# # === 読み込み ===
# fix_df = pd.read_csv(fixation_csv_path)
# time_df = pd.read_csv(timepoints_csv_path)

# # === データ空チェック ===
# if fix_df.empty:
#     print(f"⚠️ データが空です: {fixation_csv_path}")
#     exit(1)

# if time_df.empty:
#     print(f"⚠️ データが空です: {timepoints_csv_path}")
#     exit(1)

# # === 試行番号取得
# trial_number = fix_df["trial"].iloc[0]
# start_sec_row = time_df.loc[time_df["trial"] == trial_number]
# if start_sec_row.empty:
#     print(f"⚠️ 試行番号 {trial_number} に対応するstart_secが見つかりません")
#     exit(1)

# start_sec = start_sec_row["start_sec"].values[0]

# # === 相対時刻列を作成 ===
# fix_df["relative_start_sec"] = fix_df["start_time"] - start_sec

# # === 区間定義 ===
# intervals = [
#     (0.0, 0.5),
#     (0.5, 2.0),
#     (2.0, 5.0),
#     (5.0, None)  # Noneは上限なし
# ]

# # === 区間ごとに抽出・保存 ===
# for idx, (lower, upper) in enumerate(intervals, start=1):
#     if upper is not None:
#         mask = (fix_df["relative_start_sec"] >= lower) & (fix_df["relative_start_sec"] < upper)
#         suffix = f"{lower:.1f}-{upper:.1f}s"
#     else:
#         mask = (fix_df["relative_start_sec"] >= lower)
#         suffix = f"{lower:.1f}s_plus"

#     slice_df = fix_df.loc[mask].copy()

#     output_path = os.path.join(
#         output_dir,
#         f"fixation_{trial_number}_interval_{suffix}.csv"
#     )
#     slice_df.to_csv(output_path, index=False, encoding="utf-8-sig")
#     print(f"✅ 区間 {suffix} 出力: {output_path}")

# # === AOI潜時の計算 ===
# aoi_df = fix_df.loc[fix_df["AOI_label"] != "outside"]
# if aoi_df.empty:
#     latency_sec = None
#     print("⚠️ AOI内注視がありませんでした")
# else:
#     first_aoi_start = aoi_df["start_time"].min()
#     latency_sec = first_aoi_start - start_sec

# # === AOI潜時をCSVで保存 ===
# latency_output_path = os.path.join(output_dir, f"AOI_latency_trial_{trial_number}.csv")
# latency_df = pd.DataFrame([{
#     "trial": trial_number,
#     "aoi_latency_sec": latency_sec
# }])
# latency_df.to_csv(latency_output_path, index=False, encoding="utf-8-sig")
# print(f"✅ AOI潜時 出力: {latency_output_path}")


In [ ]:
import pandas as pd
import os

# === 入力ファイルパス ===
fixation_dir = "./exported_csv/fixation_counts/IVT/fixation_with_AOI_labels"
timepoints_dir = "./exported_csv/sampling_df_add_kukan"

# === 出力先フォルダ ===
output_dir = "exported_csv/fixation_sliced_by_time"
os.makedirs(output_dir, exist_ok=True)


# === 区間定義 ===
intervals = [
    (0.0, 0.5),
    (0.5, 2.0),
    (2.0, 5.0),
    (5.0, None)
]


# === 潜時を貯めるリスト
latency_records = []

# === ループ ===
for subject_id in range(1, 20):
    for experiment_id in range(1, 4):
        # timepointsファイル
        timepoints_csv_path = os.path.join(
            timepoints_dir,
            f"with_timepoints_sampling_df_id{subject_id:03}-{experiment_id:03}.csv"
        )
        if not os.path.exists(timepoints_csv_path):
            print(f"⚠️ timepointsファイルが無い: {timepoints_csv_path}")
            continue

        time_df = pd.read_csv(timepoints_csv_path)
        if time_df.empty:
            print(f"⚠️ timepointsファイルが空: {timepoints_csv_path}")
            continue

        for trial_num in range(8):
            # fixationファイル
            fixation_csv_path = os.path.join(
                fixation_dir,
                f"fixation_AOI_label_{subject_id:03}-{experiment_id:03}-{trial_num}.csv"
            )
            if not os.path.exists(fixation_csv_path):
                print(f"⚠️ fixationファイルが無い: {fixation_csv_path}")
                continue

            fix_df = pd.read_csv(fixation_csv_path)
            if fix_df.empty:
                print(f"⚠️ fixationファイルが空: {fixation_csv_path}")
                continue

            # 試行番号取得
            trial_number = fix_df["trial"].iloc[0]
            start_sec_row = time_df.loc[time_df["trial"] == trial_number]
            if start_sec_row.empty:
                print(f"⚠️ 試行 {trial_number} のstart_secが無い: {timepoints_csv_path}")
                continue

            start_sec = start_sec_row["start_sec"].values[0]

            # 相対時間
            fix_df["relative_start_sec"] = fix_df["start_time"] - start_sec

            # 区間ごとに出力
            for (lower, upper) in intervals:
                if upper is not None:
                    mask = (fix_df["relative_start_sec"] >= lower) & (fix_df["relative_start_sec"] < upper)
                    suffix = f"{lower:.1f}-{upper:.1f}s"
                else:
                    mask = (fix_df["relative_start_sec"] >= lower)
                    suffix = f"{lower:.1f}s_plus"

                slice_df = fix_df.loc[mask].copy()

                output_path = os.path.join(
                    output_dir,
                    f"fixation_{subject_id:03}-{experiment_id:03}-{trial_num}_interval_{suffix}.csv"
                )
                slice_df.to_csv(output_path, index=False, encoding="utf-8-sig")

            # AOI潜時
            aoi_df = fix_df.loc[fix_df["AOI_label"] != "outside"]
            if aoi_df.empty:
                latency_sec = None
            else:
                first_aoi_start = aoi_df["start_time"].min()
                latency_sec = first_aoi_start - start_sec

            # 潜時をリストに記録
            latency_records.append({
                "subject_id": subject_id,
                "experiment_id": experiment_id,
                "trial": trial_num,
                "aoi_latency_sec": latency_sec
            })

            print(f"✅ 完了: subj={subject_id}, exp={experiment_id}, trial={trial_num}")

# === 潜時を1ファイルにまとめて出力
latency_df = pd.DataFrame(latency_records)
latency_output_path = os.path.join(output_dir, "AOI_latency_all_trials.csv")
latency_df.to_csv(latency_output_path, index=False, encoding="utf-8-sig")
print(f"✅ AOI潜時 全試行分 出力: {latency_output_path}")